# 94 - Introducción a LangChain — Base

LangChain permite crear **cadenas de procesamiento** que combinan modelos y pasos secuenciales.
Aquí usamos un ejemplo simulado sin LLM real.

In [ ]:
def simple_chain(question):
    step1=f'Pregunta estandarizada: {question.strip().capitalize()}?'
    step2=f'Respuesta simulada a "{question}": Esto es un ejemplo.'
    return step1+'\n'+step2

print(simple_chain('qué es la ia'))


### Ejercicio
1. Crea un chain que convierta texto en minúsculas → mayúsculas → invertir orden.
2. Explica cómo encadenar pasos permite estructurar pipelines complejos.

## Ampliación progresiva

LangChain se entiende mejor si pensamos en piezas pequeñas conectadas. Antes de usar un LLM real, podemos simular estos conceptos:

1. Un paso transforma datos.
2. Una cadena guarda trazas intermedias.
3. Un paso puede decidir qué ruta seguir.
4. La respuesta puede enriquecerse con contexto local.

Esta celda construye una cadena genérica de pasos y guarda una traza de las transformaciones aplicadas. Así se puede depurar qué entra y qué sale de cada fase del pipeline.


In [ ]:
def run_chain(text, steps):
    trace = []
    value = text
    for name, fn in steps:
        value = fn(value)
        trace.append({"step": name, "value": value})
    return value, trace

steps = [
    ("normalizar", lambda s: " ".join(s.strip().split())),
    ("pregunta", lambda s: s if s.endswith("?") else s + "?"),
    ("plantilla", lambda s: f"Responde en 3 líneas: {s}"),
]

result, trace = run_chain("  que es una red neuronal  ", steps)
print(result)
print(trace)

Esta celda añade una base de conocimiento local y un pequeño enrutador de intenciones. Sirve para simular cómo una cadena decide si debe recuperar contexto, responder de forma general o derivar la pregunta a otra rama.


In [ ]:
knowledge_base = {
    "red neuronal": "Modelo inspirado en capas de neuronas artificiales.",
    "regresión logística": "Clasificador lineal usado para clasificación binaria o multiclase.",
    "overfitting": "Problema donde el modelo memoriza el entrenamiento y generaliza mal.",
}

def retrieve_local_context(question):
    q = question.lower()
    matches = [text for key, text in knowledge_base.items() if key in q]
    return matches or ["No hay contexto local; responder con cautela."]

def qa_chain(question):
    clean, trace = run_chain(question, steps[:2])
    context = retrieve_local_context(clean)
    answer = f"Pregunta: {clean}\nContexto: {context[0]}\nRespuesta simulada: usa el contexto para contestar."
    return answer, trace + [{"step": "contexto", "value": context}]

answer, qa_trace = qa_chain("¿Qué es una red neuronal?")
print(answer)
print(qa_trace)

### Reto adicional

Crea una rama condicional: si la pregunta contiene `compara`, devuelve una plantilla de comparación; si contiene `define`, devuelve una definición breve.

## Para profundizar

Este notebook muestra el patrón básico de cadenas en LangChain. La **documentación completa** (`LangChain_Documentacion.md`) incluye:

- **Prompt Templates** con FewShot, examples dinámicos.
- **Chat Models** (OpenAI, Anthropic, Ollama, Azure).
- **Chains** complejas: SequentialChain, RouterChain.
- **Tools** para acciones externas (Wikipedia, búsqueda web, calculadora).
- **Memory** (buffer, summary, window).
- **Retrievers** y RAG con Chroma, FAISS, Pinecone.
- **Agents** que deciden qué herramienta usar.
- **Output Parsers** para estructurar respuestas.
- **Error handling** y fallbacks.

Consulta el documento en la carpeta `Documentacion/` para ver ejemplos de estas funcionalidades avanzadas.

In [ ]:
# Ejemplo de Tool (patrón, necesita langchain installed)
# from langchain.tools import Tool
# def buscar_wikipedia(query):
#     return wikipedia.search(query)
# tool = Tool(name="wikipedia", func=buscar_wikipedia, description="Útil para buscar información factual")

# Ejemplo de Memory (patrón)
# from langchain.memory import ConversationBufferMemory
# memory = ConversationBufferMemory()
# memory.save_context({"input": "Hola"}, {"output": "Hola, soy un asistente"})
# Para ver más: consulta LangChain_Documentacion.md sección 'Compoñentes reais'